# 04 — Track-1 production: topic lists from every hearing summary

**Purpose.** Map each of the 2,352 Stage-2 final summaries to a ranked list
of at most five neutral topic phrases — one call per hearing, no stance
field. Track 1 covers stance-free attention (topics discussed without
directional positions), feeds the pooled phrase set for taxonomy induction
alongside the Track-2 record topics, and anchors the Track-1 vs. Track-2
convergence check (plan §Track 1, §internal validation).

**Prompt.** `summary-topics-0.2-dev`, run at scale on the researcher's
decision of 2026-08-22 after two dev iterations on 28 seeded summaries
closed the observed failure classes (bare entities as topics, bare bill
numbers, tail padding). The topic-phrase grammar is carried verbatim from
the Track-2 prompt so the two phrase pools obey one set of rules.

**Logging.** Every call appends one JSONL line to
`status/extract.jsonl` — full prompt, full output, token counts, seconds —
before the output is parsed, so a call that happened is on disk even if
everything after it fails. The verification cell checks the log against the
output table.

**Resumable.** A hearing is done when its row is in the output parquet,
which is rewritten every 100 hearings; a re-run skips finished hearings.

**Inputs.** `data/cleaned/01_text_summarization/hearing_summaries.parquet`
(Stage-2 assembly, notebook 02/04).

**Outputs.** `interim/topics/track1/topics.parquet` — one row per hearing
with the ranked topics serialized; `interim/topics/track1/status/extract.jsonl`.

**Execution.** Run inside a GPU allocation (researcher decision 2026-08-22:
the ~2,352-call, sub-2-hour job runs in the interactive session rather than
as a batch submission; `jupyter nbconvert --execute --inplace`).

## Setup and parameters

Same environment discipline as the Track-2 notebooks: weights on scratch,
token from the gitignored `.env`, generation settings from `summarize` so
every stage samples identically. 256 new tokens is ample for five short
phrases; the cap being hit is recorded as an error, not hidden.

In [1]:
import hashlib
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, "../lib")

from paths import HF_CACHE, topics_run_dir

HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
for line in (Path(".env").read_text().splitlines() if Path(".env").exists() else []):
    if line.startswith("HF_TOKEN="):
        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import stance
import summarize as S

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
MAX_NEW_TOKENS = 256

SUMMARIES = Path("../../../data/cleaned/01_text_summarization/hearing_summaries.parquet")
TRACK1 = topics_run_dir("track1")
OUT_FILE = TRACK1 / "topics.parquet"
LOG_FILE = TRACK1 / "status" / "extract.jsonl"

print(f"model   : {MODEL}")
print(f"prompt  : {stance.TRACK1_PROMPT_VERSION}")
print(f"cap     : {MAX_NEW_TOKENS} new tokens, at most {stance.TRACK1_MAX_TOPICS} topics")
print(f"sampling: {S.GENERATION}")
print(f"gpu     : {torch.cuda.get_device_name(0)}")

model   : Qwen/Qwen3.5-4B
prompt  : summary-topics-0.2-dev
cap     : 256 new tokens, at most 5 topics
sampling: {'do_sample': True, 'temperature': 0.1}
gpu     : NVIDIA A40


## Step 1 — Worklist

Every hearing with a Stage-2 summary (2,352; the assembly notebook verified
all carry status `ok`). Finished hearings are read back from the output
parquet, so a re-run only touches what is missing.

In [2]:
summaries = pd.read_parquet(
    SUMMARIES, columns=["packageId", "year", "chamber", "title", "summary", "status"])
assert (summaries["status"] == "ok").all(), "Stage-2 left non-ok summaries"

rows = []
if OUT_FILE.exists():
    rows = pd.read_parquet(OUT_FILE).to_dict("records")
done = {r["packageId"] for r in rows}
todo = summaries.loc[~summaries["packageId"].isin(done)].reset_index(drop=True)

print(f"hearings  : {len(summaries):,}")
print(f"  done    : {len(done):,}")
print(f"  to run  : {len(todo):,}")

hearings  : 2,352
  done    : 0
  to run  : 2,352


## Step 2 — Load the model

In [3]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s — "
      f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded in 4s — 4.21B params


## Step 3 — Extract

Per hearing: seed from the packageId hash (order-free bit reproducibility),
one call, log the call **before** parsing it, validate with `parse_topics`.
The output parquet is rewritten every 100 hearings, so an interrupted run
loses at most a few minutes of work.

In [4]:
started = time.time()
for n, row in enumerate(todo.itertuples(), 1):
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(row.packageId.encode()).digest()[:4], "big"))
    log = S.CallLog(LOG_FILE, row.packageId, MODEL,
                    prompt_version=stance.TRACK1_PROMPT_VERSION)

    user = stance.render_track1(row.summary)
    t0 = time.time()
    output, hit_cap = S._generate(model, tokenizer, stance.SYSTEM, user,
                                   max_new_tokens=MAX_NEW_TOKENS)
    log.write("track1", 0, 0, 1, user, output, tokenizer, time.time() - t0)
    topics, errors = stance.parse_topics(output)
    if hit_cap:
        errors.append("hit_token_cap")

    rows.append({
        "packageId": row.packageId, "year": int(row.year), "chamber": row.chamber,
        "prompt_version": stance.TRACK1_PROMPT_VERSION, "model": MODEL,
        "n_topics": len(topics),
        "topics_json": json.dumps(topics),
        "errors": ";".join(errors),
    })
    if n % 100 == 0 or n == len(todo):
        pd.DataFrame(rows).to_parquet(OUT_FILE, compression="zstd", index=False)
        rate = n / (time.time() - started)
        print(f"  {n:>5,}/{len(todo):,}  {rate:.2f} hearings/s  "
              f"eta {(len(todo) - n) / rate / 60:.0f} min")

pd.DataFrame(rows).to_parquet(OUT_FILE, compression="zstd", index=False)
print(f"\n{len(rows):,} hearings, {(time.time() - started) / 60:.0f} min this session")

    100/2,352  0.42 hearings/s  eta 88 min


    200/2,352  0.41 hearings/s  eta 87 min


    300/2,352  0.41 hearings/s  eta 83 min


    400/2,352  0.41 hearings/s  eta 80 min


    500/2,352  0.41 hearings/s  eta 76 min


    600/2,352  0.41 hearings/s  eta 72 min


... (14 outputs left out)

  2,100/2,352  0.41 hearings/s  eta 10 min


  2,200/2,352  0.41 hearings/s  eta 6 min


  2,300/2,352  0.41 hearings/s  eta 2 min


  2,352/2,352  0.41 hearings/s  eta 0 min

2,352 hearings, 96 min this session


## Verification

Checked against the dev-run bands (28/28 clean, 4–5 topics/hearing) and
against the call log itself: every output row must be backed by at least
one logged call — an unlogged result would be a provenance hole, which is
exactly what the log exists to rule out.

In [5]:
back = pd.read_parquet(OUT_FILE)
assert not back.duplicated(["packageId"]).any(), "hearing done twice"
assert len(back) == len(summaries), "coverage incomplete"

logged_ids = set()
with open(LOG_FILE) as f:
    n_log_lines = 0
    for line in f:
        n_log_lines += 1
        logged_ids.add(json.loads(line)["document_id"])
unlogged = set(back["packageId"]) - logged_ids
assert not unlogged, f"rows with no logged call: {sorted(unlogged)[:5]}"

err_rate = (back["errors"] != "").mean()
all_topics = [t for js in back["topics_json"] for t in json.loads(js)]
lint = sum(1 for t in all_topics if t.get("topic_lint"))

print(f"1) hearings       : {len(back):,} of {len(summaries):,}, no duplicates")
print(f"2) call log       : {n_log_lines:,} lines, every row backed by a logged call")
print(f"3) error rate     : {err_rate:.1%} "
      f"[{'OK' if err_rate < 0.05 else 'OUT OF BAND — inspect'}]")
print(f"4) topics/hearing : {back['n_topics'].value_counts().sort_index().to_dict()}")
print(f"5) total phrases  : {len(all_topics):,} for the pooled phrase set")
print(f"6) topic lint     : {lint:,} flagged "
      f"({lint / max(len(all_topics), 1):.1%}) — review queue, not removal")
print(f"7) prompt version : {back['prompt_version'].unique().tolist()}")

1) hearings       : 2,352 of 2,352, no duplicates
2) call log       : 2,352 lines, every row backed by a logged call
3) error rate     : 0.6% [OK]
4) topics/hearing : {2: 2, 3: 9, 4: 144, 5: 2197}
5) total phrases  : 11,592 for the pooled phrase set
6) topic lint     : 219 flagged (1.9%) — review queue, not removal
7) prompt version : ['summary-topics-0.2-dev']
